# 03 – Proposed method: night-specific random-forest downscaling

The model learns, from ECOSTRESS nights, how much each 70 m pixel deviates from its 980 m block mean.
At prediction time that anomaly is added to the MODIS block value and re-centred, so the block mean stays equal to MODIS (the same residual correction as TsHARP).

Compared on the same clear nights and the same ECOSTRESS reference:

| Method | Inputs |
|---|---|
| MODIS upsampled | MODIS only |
| TsHARP (Agam et al. 2007) | MODIS + Sentinel-2 NDVI |
| Sentinel-2-only random forest | MODIS + Sentinel-2 NDVI and NDBI |
| **Proposed** | MODIS + vegetation, built-up, location, daytime heat storage, night context |

Validation: leave-one-night-out for the main comparison; 6-fold leave-nights-out for the ablation.

Outputs: `data/results_main.csv`, `data/results_ablation.csv`, `figures/03_*.png`.

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/KeremBasimc/istanbul-night-heat-island"
if "google.colab" in sys.modules or Path("/content").exists():
    ROOT = Path("/content/istanbul-night-heat-island")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data"
FIGS = ROOT / "figures"
DATA.mkdir(exist_ok=True); FIGS.mkdir(exist_ok=True)

import earthaccess
# Tries, in order: environment variables, ~/.netrc, then an interactive prompt.
print("Earthdata login ok:", earthaccess.login().authenticated)

## Clear nights and grids

In [ ]:
import time
import numpy as np
import pandas as pd
from nhi import data, downscale as ds, features as fe, model as M

nights_csv = DATA / "ecostress_nights.csv"
if nights_csv.exists():
    per_night = pd.read_csv(nights_csv, parse_dates=["local"])
else:
    g = data.search_ecostress("2018-07-01", "2025-09-30")
    g = data.add_clear_fraction(g[g.is_night], DATA / "ecostress")
    per_night = data.best_granule_per_night(g)
    per_night.to_csv(nights_csv, index=False)
clear = per_night[per_night.clear_frac >= 0.7].reset_index(drop=True)
coarse, fine = data.grids()
print(len(clear), "clear nights")

## Static predictor layers (cached per year)

In [ ]:
t0 = time.time()
wc = fe.worldcover_fractions(fine)
sea_km = fe.distance_to_sea(wc["water"]) / 1000
elev = fe.elevation(fine)
land = wc["water"] < 0.5

def static_for(year):
    path = DATA / f"static_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    s2 = fe.sentinel2_indices(year, fine)
    layers = {"ndvi": s2["ndvi"], "ndbi": s2["ndbi"],
              "tree": wc["tree"], "grass": wc["grass"], "shrub": wc["shrub"], "crop": wc["crop"],
              "built": wc["built"], "bare": wc["bare"],
              "sea_dist_km": sea_km, "elev": elev,
              "landsat_day": fe.landsat_day_lst(year, fine)}
    np.savez_compressed(path, **layers)
    return layers

years = sorted(pd.to_datetime(clear.night_date).dt.year.unique())
STATIC = {y: static_for(y) for y in years}
bldg_h = fe.ghsl_building_height(fine, folder=DATA / "ghsl")   # GHSL 2018, same for all years
for y in years:
    STATIC[y]["bldg_height"] = bldg_h
STATIC_DEV = {y: M.static_with_deviations(STATIC[y]) for y in years}
print("static layers for", years, f"in {time.time() - t0:.0f} s")

## Assemble every clear night and score the two non-learning benchmarks

In [ ]:
t0 = time.time()
NIGHTS, bench = [], []
for r in clear.itertuples():
    paths = data.download([r.url_LST, r.url_cloud, r.url_water], DATA / "ecostress")
    eco = np.where(land, data.ecostress_on_grid(*paths, fine).values, np.nan)   # same land mask for all methods
    passes = data.load_modis_night_passes(pd.Timestamp(r.night_date).date(), coarse, hdf_folder=DATA / "modis_hdf")
    pick = data.closest_modis_pass(passes, pd.Timestamp(r.local))
    if pick is None:
        continue
    platform, mp, gap = pick
    modis_c = mp["lst"].values.astype("float32")
    year = pd.Timestamp(r.night_date).year
    layers = M.night_layers(STATIC_DEV[year], modis_c, pd.Timestamp(r.local), gap)
    NIGHTS.append({"key": str(r.night_date), "layers": layers, "y": M.anomaly_target(eco),
                   "eco": eco, "modis_coarse": modis_c, "land": land,
                   "platform": platform, "gap_h": gap, "local": r.local})
    ndvi_land = np.where(land, STATIC[year]["ndvi"], np.nan)
    for name, pred in [("MODIS upsampled", np.where(land, ds.upsample(modis_c), np.nan)),
                       ("TsHARP", ds.tsharp(modis_c, ndvi_land)[0])]:
        m = ds.metrics(pred, eco); m.update(night_date=str(r.night_date), method=name)
        bench.append(m)
print(len(NIGHTS), f"nights assembled in {time.time() - t0:.0f} s")

## Leave-one-night-out: proposed model and the Sentinel-2-only random forest

In [ ]:
t0 = time.time()
proposed = M.cross_validate(NIGHTS, M.features_for()).assign(method="Proposed")
print(f"proposed: {time.time() - t0:.0f} s"); t0 = time.time()
s2only = M.cross_validate(NIGHTS, M.S2_ONLY).assign(method="S2-only RF")
print(f"S2-only RF: {time.time() - t0:.0f} s")

meta = pd.DataFrame([{"night_date": n["key"], "platform": n["platform"], "gap_h": n["gap_h"],
                      "month": pd.Timestamp(n["local"]).month} for n in NIGHTS])
res = pd.concat([pd.DataFrame(bench), s2only, proposed]).merge(meta, on="night_date")
res.to_csv(DATA / "results_main.csv", index=False)
order = ["MODIS upsampled", "TsHARP", "S2-only RF", "Proposed"]
summary = res.groupby("method")[["rmse", "crmse", "mae", "bias", "r2", "r"]].mean().loc[order].round(2)
summary

## Is the gain over TsHARP real? Paired comparison with a bootstrap over nights

In [ ]:
def paired(a, b, col, n_boot=5000, seed=0):
    pa = res[res.method == a].set_index("night_date")[col]
    pb = res[res.method == b].set_index("night_date")[col]
    d = (pa - pb).dropna().values
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, (n_boot, d.size), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"comparison": f"{a} - {b}", "metric": col, "mean_diff": d.mean(), "ci_low": lo,
            "ci_high": hi, "nights_better": int((d < 0).sum()), "nights": d.size}

tests = pd.DataFrame([paired("Proposed", "TsHARP", c) for c in ("crmse", "rmse")] +
                     [paired("Proposed", "S2-only RF", c) for c in ("crmse", "rmse")])
tests.round(3)

## Ablation (6-fold leave-nights-out): which predictor group matters?

In [ ]:
t0 = time.time()
groups = list(M.STATIC_GROUPS)
variants = {"all": M.features_for()}
for g in groups:
    variants[f"without {g}"] = M.features_for([x for x in groups if x != g])
variants["without night context"] = M.features_for(night_context=False)
abl = []
for name, feats in variants.items():
    r = M.cross_validate(NIGHTS, feats, n_folds=6)
    abl.append({"variant": name, "crmse": r.crmse.mean(), "rmse": r.rmse.mean(), "r": r.r.mean()})
abl = pd.DataFrame(abl)
abl["crmse_increase"] = abl.crmse - abl.loc[abl.variant == "all", "crmse"].item()
abl.to_csv(DATA / "results_ablation.csv", index=False)
print(f"ablation: {time.time() - t0:.0f} s")
abl.round(3)

## Feature importance of the full model (fitted on all nights)

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
names = M.features_for()
Xs, ys = zip(*[M.training_sample(n["layers"], names, n["y"], n["land"], 2000, rng) for n in NIGHTS])
full = M.make_model().fit(np.concatenate(Xs), np.concatenate(ys))
_, cols = M.design_matrix(NIGHTS[0]["layers"], names)
imp = pd.Series(full.feature_importances_, index=cols)
imp = imp.groupby(lambda c: c.removesuffix("_dev")).sum().sort_values()
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
imp.plot.barh(ax=ax); ax.set_xlabel("Impurity importance (feature + block deviation)")
ax.set_title("What explains night LST within a 980 m block?")
fig.savefig(FIGS / "03_feature_importance.png", dpi=150)
plt.show()

## Error by method and an example night

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, col, label in zip(axes, ["crmse", "rmse"], ["Bias-removed RMSE (°C)", "RMSE (°C)"]):
    ax.boxplot([res.loc[res.method == m, col].dropna() for m in order])
    ax.set_xticks(range(1, len(order) + 1), order, rotation=15)
    ax.set_title(label); ax.grid(axis="y", alpha=.3)
fig.suptitle(f"Leave-one-night-out against ECOSTRESS, {len(NIGHTS)} clear nights")
fig.savefig(FIGS / "03_method_comparison.png", dpi=150)
plt.show()

# Example: the clearest summer night
summer = [n for n in NIGHTS if pd.Timestamp(n["local"]).month in (6, 7, 8, 9)] or NIGHTS
ex = max(summer, key=lambda n: np.isfinite(n["eco"]).sum())
others = [n for n in NIGHTS if n["key"] != ex["key"]]
Xs, ys = zip(*[M.training_sample(n["layers"], names, n["y"], n["land"], 2000, rng) for n in others])
m_ex = M.make_model().fit(np.concatenate(Xs), np.concatenate(ys))
pred = M.predict_night(m_ex, ex["layers"], names, ex["modis_coarse"], ex["land"])
year = pd.Timestamp(ex["key"]).year
tsh = ds.tsharp(ex["modis_coarse"], np.where(land, STATIC[year]["ndvi"], np.nan))[0]
vmin, vmax = np.nanpercentile(ex["eco"], [2, 98])
fig, axes = plt.subplots(2, 2, figsize=(13, 7.5), constrained_layout=True)
for ax, arr, title in [(axes[0, 0], ex["eco"], "ECOSTRESS 70 m (reference, not used for this night)"),
                       (axes[0, 1], np.where(land, ds.upsample(ex["modis_coarse"]), np.nan), f"MODIS {ex['platform'].title()} 980 m"),
                       (axes[1, 0], tsh, "TsHARP 70 m"),
                       (axes[1, 1], np.where(land, pred, np.nan), "Proposed 70 m")]:
    im = ax.imshow(arr, cmap="inferno", vmin=vmin, vmax=vmax)
    mt = ds.metrics(arr, ex["eco"]) if arr is not ex["eco"] else None
    ax.set_title(title + (f"\nbias-removed RMSE {mt['crmse']:.2f} °C, r {mt['r']:.2f}" if mt else ""))
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, label="Land surface temperature (°C)", shrink=0.8)
fig.suptitle(f"Night of {ex['key']} (ECOSTRESS {pd.Timestamp(ex['local']):%H:%M}, MODIS gap {ex['gap_h']:.1f} h)")
fig.savefig(FIGS / "03_example_night.png", dpi=150)
plt.show()

## Reading the results

- `crmse` (bias-removed RMSE) and `r` measure how well the 70 m spatial pattern is reproduced; this is what downscaling adds.
- `rmse` and `bias` also contain the time gap between the MODIS pass and ECOSTRESS, which no downscaling method can remove; the night-context features only partly absorb it.
- The paired bootstrap gives the mean difference against TsHARP with a 95 % interval over nights.